# 4. Merge Clinical and Corrected GEX_MAT

In [80]:
LIMMA_GENES = True

## Read datasets

In [81]:
import polars as pl

clinical = pl.read_csv(f"../dataset/created/clinical.csv")
clinical

,patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_month,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source,pfs_label,cobimetinib,dabrafenib,trametinib,vemurafenib,C195W,K601I,MND,R558Q,V600E,V600K,V600R
i64,str,str,i64,str,str,str,f64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,"""BS_000""","""male""",56,"""IV""",null,"""normal""",30.5,"""no""","""no""","""no""","""Blateau et al.""",2,0,1,1,0,0,0,0,0,1,0,0
1,"""BS_001""","""male""",86,"""IV""",null,"""normal""",24.1,"""no""","""no""","""no""","""Blateau et al.""",2,0,1,0,0,0,0,0,0,1,0,0
2,"""BS_002""","""female""",47,"""IV""",null,"""normal""",14.1,"""no""","""no""","""no""","""Blateau et al.""",2,0,1,1,0,0,0,0,0,1,0,0
3,"""BS_003""","""female""",50,"""IV""",null,null,1.6,"""no""","""no""","""no""","""Blateau et al.""",0,0,0,0,1,0,0,0,0,1,0,0
4,"""BS_004""","""female""",47,"""IV""",null,"""elevated""",11.9,"""no""","""no""","""no""","""Blateau et al.""",1,0,1,1,0,0,0,0,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
410,"""HL_Shi-40""","""male""",47,"""IV""","""M1C""",null,3.0,"""no""","""no""","""no""","""Hugo et al.""",0,0,0,0,1,0,0,0,0,1,0,0
411,"""HL_Shi-41""","""male""",39,"""IV""","""M1A""",null,4.0,"""no""","""no""","""no""","""Hugo et al.""",0,0,0,0,1,0,0,0,0,1,0,0
412,"""HL_Shi-42""","""male""",84,"""IV""","""M1C""",null,8.0,"""no""","""no""","""no""","""Hugo et al.""",1,0,1,0,0,0,0,0,0,1,0,0


In [82]:
if LIMMA_GENES == True:
    gex_mat = pl.read_csv(f"../dataset/created/gex_mat_corrected_sig.csv")
    print(gex_mat)
else:
    gex_mat = pl.read_csv(f"../dataset/created/gex_mat_corrected.csv")
    # 1. Define which columns are the numeric data (samples)
    # We exclude 'HGNC' because it is a string/label
    sample_cols = [col for col in gex_mat.columns if col != 'HGNC']

    # 2. Compute variance for each row and add it as a new column
    # We use horizontal variance across the sample columns
    gex_with_var = gex_mat.with_columns(
        var = pl.concat_list(sample_cols).list.var()
    )

    # 3. Sort by variance descending and take the top 1000
    most_variable_genes = gex_with_var.sort("var", descending=True).head(1000)

    # 4. (Optional) Drop the 'var' column if you don't need it anymore
    gex_mat = most_variable_genes.drop("var")
    print(gex_mat)

shape: (122, 138)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ HGNC   ┆ 8755_035H  ┆ Pt1-baseli ┆ 08766     ┆ … ┆ 05420129C ┆ Pt2-basel ┆ 05320205B ┆ 05320100B │
│ ---    ┆ PreB       ┆ ne         ┆ PreC_014G ┆   ┆ ---       ┆ ine       ┆ ---       ┆ ---       │
│ str    ┆ ---        ┆ ---        ┆ ---       ┆   ┆ f64       ┆ ---       ┆ f64       ┆ f64       │
│        ┆ f64        ┆ f64        ┆ f64       ┆   ┆           ┆ f64       ┆           ┆           │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ MAGEA4 ┆ 4.483408   ┆ 4.617514   ┆ 3.047771  ┆ … ┆ 4.332853  ┆ 4.636228  ┆ 4.329003  ┆ 5.337362  │
│ VCX3A  ┆ 10.136349  ┆ 5.529758   ┆ 4.977531  ┆ … ┆ 5.13136   ┆ 5.50946   ┆ 5.12214   ┆ 7.185458  │
│ SV2B   ┆ 4.867191   ┆ 4.884898   ┆ 3.475491  ┆ … ┆ 4.680057  ┆ 4.889576  ┆ 4.730536  ┆ 4.687803  │
│ MAGEA1 ┆ 8.881613   ┆ 4.765432   ┆ 3.278967  ┆ … ┆ 4.5101    ┆ 4.764475

In [83]:
sample_source = pl.read_csv(f"../dataset/created/sample_source.csv")
sample_source

sample_id,source,patientID,pfs_label
str,str,str,i64
"""8755_035H PreB""","""Rizos et al.""","""RL_WMD-021""",1
"""Pt1-baseline""","""Hugo et al.""","""HL_Shi-15""",1
"""08766 PreC_014G""","""Long et al.""","""LR_WMD-022""",0
"""05320220B""","""Yan et al.""","""YR_2148""",1
"""03660598B""","""Yan et al.""","""YR_3708""",2
…,…,…,…
"""05320216B""","""Yan et al.""","""YR_2220""",0
"""05420129C""","""Yan et al.""","""YR_102011""",0
"""Pt2-baseline""","""Hugo et al.""","""HL_Shi-26""",0


## Convert GEX_MAT to patient-wised

In [84]:
gex_info = gex_mat.transpose(column_names='HGNC').insert_column(
    0, pl.Series('sample_id', gex_mat.columns[1:])
)
gex_info

sample_id,MAGEA4,VCX3A,SV2B,MAGEA1,VCX2,BAIAP2L1,MAGEA10,CTAG2,CSAG2,CKMT1A,GNB5,HES6,CTNNA3,VCY,COMMD9,PTPMT1,POLN,XAGE1B,CHRAC1,CKMT1B,CD1D,MAGEA12,SCRG1,CAMK2N1,LOC497256,LHX2,ERICH1,DGCR5,XAGE1A,RBPMS2,PPP1R1C,SEMA3G,SLC31A1,MAGEA9B,TRAF6,CSAG3,…,DSCR8,C1QTNF9B,PPP1R3F,SDS,NUDT15,DENND5A,RAB11FIP1,ZNF395,TUBB3,TIGD6,AIF1L,ZNF229,LOC728554,TCFL5,PLAG1,TDH,VLDLR,ARHGAP28,OPHN1,RAD9A,FBXO48,TIAM1,GAP43,SURF6,MYOM2,CSAG1,CRYZ,LRFN2,MAGEA3,ABHD6,ARL10,DNHD1,PPIH,TFF3,PBK,FGF13,EXOSC3
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""8755_035H PreB""",4.483408,10.136349,4.867191,8.881613,8.81662,4.158633,9.926213,9.88864,7.125749,3.866415,3.942228,5.900993,3.915326,8.499373,6.830646,7.610253,5.608627,3.851377,5.192277,3.421401,5.599627,7.795181,6.742839,5.425703,3.748532,5.844848,4.981538,3.87949,4.691677,5.750853,6.044121,5.530191,5.670115,4.426608,5.155872,6.486161,…,8.908763,4.271362,3.62584,6.169884,4.843995,7.204801,6.176078,6.274013,8.18257,4.105381,7.698154,5.446986,6.317401,4.558975,5.617931,3.760208,4.458934,5.630959,4.166247,6.117598,4.042257,4.377803,5.486901,6.300762,3.368862,8.339139,5.173394,4.239306,6.753617,4.41161,4.404474,5.127989,7.63735,2.789269,3.810405,4.723208,5.856198
"""Pt1-baseline""",4.617514,5.529758,4.884898,4.765432,5.637169,5.938362,4.813479,4.111068,5.115084,4.356327,5.237074,6.130015,3.961018,5.244908,6.295192,6.706846,4.590674,6.625461,5.349656,4.008229,4.909141,5.11004,5.373944,5.661283,4.216822,5.395882,6.641997,4.207781,7.525156,5.639509,4.361478,3.958167,5.985786,4.591202,5.222628,5.285646,…,5.065562,4.528146,4.968059,4.615695,5.658829,6.815835,6.485807,6.570388,6.965012,4.693085,7.835245,4.222007,6.369709,6.356159,5.189388,3.841569,5.904396,4.824758,4.639857,5.410851,4.230491,5.230425,4.303065,6.801718,4.724729,5.556659,6.34834,3.960911,4.567013,5.369927,4.39844,4.719416,6.541229,6.148049,5.847441,5.741465,6.665037
"""08766 PreC_014G""",3.047771,4.977531,3.475491,3.278967,3.935683,6.5266,1.739232,4.361025,2.21075,8.161308,5.561452,6.81307,3.931472,3.511027,5.839293,6.296091,3.507599,3.407724,5.880657,7.799422,3.382025,3.149098,6.428041,6.418649,3.725129,5.735883,6.829744,5.83814,4.467828,6.128533,3.51699,3.923536,6.342268,4.182003,4.567562,3.969568,…,3.002705,5.201066,5.918696,4.587897,6.329254,6.61739,6.51363,6.638589,7.95737,4.943513,8.432018,4.316272,7.490388,7.330809,5.260329,3.577529,4.920243,4.657764,4.569284,5.55162,4.598806,5.759604,3.954216,6.435285,5.902513,2.748082,6.956953,3.588606,3.456867,5.883267,3.542523,4.588816,6.203982,6.440745,7.838665,3.365906,6.999273
"""05320220B""",4.329003,5.12214,4.681457,4.5101,5.284913,6.233655,4.600569,4.685542,4.710835,4.908119,5.652038,6.768127,3.992354,4.869216,6.682885,6.723418,4.405557,6.035066,5.607562,4.535833,4.81854,4.788532,5.328728,6.427116,4.124999,5.530635,5.848424,4.296672,6.88794,5.919544,4.206064,3.992522,6.819635,4.488521,5.102136,4.759547,…,4.79534,4.608212,5.097925,4.77327,5.708743,7.092949,6.276513,6.330508,7.954861,4.670516,7.617689,4.006487,6.616684,6.286267,4.828574,3.883233,6.69116,4.538565,4.713715,5.397739,4.266103,4.439508,4.319613,6.494991,4.828422,4.960341,6.98078,4.002205,5.048048,6.065623,4.022847,4.760772,7.165262,5.021874,6.107214,5.12339,6.70964
"""03660598B""",6.570555,5.158157,4.697329,5.436335,5.282412,5.724922,4.673865,4.685542,5.002026,4.823392,5.520698,6.763863,3.992659,4.913842,6.603374,6.8341,4.365546,6.036902,5.500889,4.433253,4.820562,6.195991,5.359955,6.393214,4.124999,5.557993,5.972715,4.290036,6.900869,5.960884,4.215222,3.997007,6.970775,4.488521,4.984,5.663993,…,4.716858,4.614575,5.155613,4.738068,5.780937,6.69161,6.832015,6.412755,7.987326,4.711399,7.580741,4.09871,6.617454,6.15428,4.826822,3.883233,7.394174,4.58168,4.655143,5.489278,4.252246,4.676489

In [85]:
gex = sample_source.join(other=gex_info, on='sample_id')
gex

sample_id,source,patientID,pfs_label,MAGEA4,VCX3A,SV2B,MAGEA1,VCX2,BAIAP2L1,MAGEA10,CTAG2,CSAG2,CKMT1A,GNB5,HES6,CTNNA3,VCY,COMMD9,PTPMT1,POLN,XAGE1B,CHRAC1,CKMT1B,CD1D,MAGEA12,SCRG1,CAMK2N1,LOC497256,LHX2,ERICH1,DGCR5,XAGE1A,RBPMS2,PPP1R1C,SEMA3G,SLC31A1,…,DSCR8,C1QTNF9B,PPP1R3F,SDS,NUDT15,DENND5A,RAB11FIP1,ZNF395,TUBB3,TIGD6,AIF1L,ZNF229,LOC728554,TCFL5,PLAG1,TDH,VLDLR,ARHGAP28,OPHN1,RAD9A,FBXO48,TIAM1,GAP43,SURF6,MYOM2,CSAG1,CRYZ,LRFN2,MAGEA3,ABHD6,ARL10,DNHD1,PPIH,TFF3,PBK,FGF13,EXOSC3
str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""8755_035H PreB""","""Rizos et al.""","""RL_WMD-021""",1,4.483408,10.136349,4.867191,8.881613,8.81662,4.158633,9.926213,9.88864,7.125749,3.866415,3.942228,5.900993,3.915326,8.499373,6.830646,7.610253,5.608627,3.851377,5.192277,3.421401,5.599627,7.795181,6.742839,5.425703,3.748532,5.844848,4.981538,3.87949,4.691677,5.750853,6.044121,5.530191,5.670115,…,8.908763,4.271362,3.62584,6.169884,4.843995,7.204801,6.176078,6.274013,8.18257,4.105381,7.698154,5.446986,6.317401,4.558975,5.617931,3.760208,4.458934,5.630959,4.166247,6.117598,4.042257,4.377803,5.486901,6.300762,3.368862,8.339139,5.173394,4.239306,6.753617,4.41161,4.404474,5.127989,7.63735,2.789269,3.810405,4.723208,5.856198
"""Pt1-baseline""","""Hugo et al.""","""HL_Shi-15""",1,4.617514,5.529758,4.884898,4.765432,5.637169,5.938362,4.813479,4.111068,5.115084,4.356327,5.237074,6.130015,3.961018,5.244908,6.295192,6.706846,4.590674,6.625461,5.349656,4.008229,4.909141,5.11004,5.373944,5.661283,4.216822,5.395882,6.641997,4.207781,7.525156,5.639509,4.361478,3.958167,5.985786,…,5.065562,4.528146,4.968059,4.615695,5.658829,6.815835,6.485807,6.570388,6.965012,4.693085,7.835245,4.222007,6.369709,6.356159,5.189388,3.841569,5.904396,4.824758,4.639857,5.410851,4.230491,5.230425,4.303065,6.801718,4.724729,5.556659,6.34834,3.960911,4.567013,5.369927,4.39844,4.719416,6.541229,6.148049,5.847441,5.741465,6.665037
"""08766 PreC_014G""","""Long et al.""","""LR_WMD-022""",0,3.047771,4.977531,3.475491,3.278967,3.935683,6.5266,1.739232,4.361025,2.21075,8.161308,5.561452,6.81307,3.931472,3.511027,5.839293,6.296091,3.507599,3.407724,5.880657,7.799422,3.382025,3.149098,6.428041,6.418649,3.725129,5.735883,6.829744,5.83814,4.467828,6.128533,3.51699,3.923536,6.342268,…,3.002705,5.201066,5.918696,4.587897,6.329254,6.61739,6.51363,6.638589,7.95737,4.943513,8.432018,4.316272,7.490388,7.330809,5.260329,3.577529,4.920243,4.657764,4.569284,5.55162,4.598806,5.759604,3.954216,6.435285,5.902513,2.748082,6.956953,3.588606,3.456867,5.883267,3.542523,4.588816,6.203982,6.440745,7.838665,3.365906,6.999273
"""05320220B""","""Yan et al.""","""YR_2148""",1,4.329003,5.12214,4.681457,4.5101,5.284913,6.233655,4.600569,4.685542,4.710835,4.908119,5.652038,6.768127,3.992354,4.869216,6.682885,6.723418,4.405557,6.035066,5.607562,4.535833,4.81854,4.788532,5.328728,6.427116,4.124999,5.530635,5.848424,4.296672,6.88794,5.919544,4.206064,3.992522,6.819635,…,4.79534,4.608212,5.097925,4.77327,5.708743,7.092949,6.276513,6.330508,7.954861,4.670516,7.617689,4.006487,6.616684,6.286267,4.828574,3.883233,6.69116,4.538565,4.713715,5.397739,4.266103,4.439508,4.319613,6.494991,4.828422,4.960341,6.98078,4.002205,5.048048,6.065623,4.022847,4.760772,7.165262,5.021874,6.107214,5.12339,6.70964
"""03660598B""","""Yan et al.""","""YR_3708""",2,6.570555,5.158157,4.697329,5.436335,5.282412,5.724922,4.673865,4.685542,5.002026,4.823392,5.520698,6.763863,3.992659,4.913842,6.603374,6.8341,4.365546,6.036902,5.500889,4.433253,4.820562,6.195991,5.359955,6.393214,4.124999,5.557993,5.972715,4.290036,6.900869,5.960884,4.215222,3.997007,6.970775,…,4.716858,4.614575,5.155613,4.738068,5.780937,6.69161,6.832015,6.412755,7.987326,4.711399,7.580741,4.09871,6.617454,6.15428,4.826822,3.883233,

## Merge with Clinical dataset

In [86]:
clinical_gex = clinical.join(gex, on='patientID', how='inner')[:, 1:]
clinical_gex

patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_month,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source,pfs_label,cobimetinib,dabrafenib,trametinib,vemurafenib,C195W,K601I,MND,R558Q,V600E,V600K,V600R,sample_id,source_right,pfs_label_right,MAGEA4,VCX3A,SV2B,MAGEA1,VCX2,BAIAP2L1,MAGEA10,CTAG2,CSAG2,CKMT1A,GNB5,…,DSCR8,C1QTNF9B,PPP1R3F,SDS,NUDT15,DENND5A,RAB11FIP1,ZNF395,TUBB3,TIGD6,AIF1L,ZNF229,LOC728554,TCFL5,PLAG1,TDH,VLDLR,ARHGAP28,OPHN1,RAD9A,FBXO48,TIAM1,GAP43,SURF6,MYOM2,CSAG1,CRYZ,LRFN2,MAGEA3,ABHD6,ARL10,DNHD1,PPIH,TFF3,PBK,FGF13,EXOSC3
str,str,i64,str,str,str,f64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""YR_3053""","""male""",26,"""IV""","""M1B""","""elevated""",2.9,null,"""no""","""no""","""Yan et al.""",0,0,0,0,1,0,0,0,0,1,0,0,"""03660555B""","""Yan et al.""",0,4.352461,5.136899,4.68535,5.189127,5.30349,5.596766,4.657524,4.691475,4.815777,4.838144,5.511944,…,4.998998,4.598046,5.108318,4.741692,5.706387,6.517363,6.42378,6.5609,7.447856,4.632723,7.46384,4.205274,6.618716,6.006904,4.923222,3.883233,6.47874,4.627395,4.733492,5.385977,4.278327,4.794856,4.396161,6.479374,4.909547,7.020221,6.370399,4.00089,6.28746,5.764514,4.015359,4.416253,6.739738,5.003677,5.890783,5.189148,6.328
"""YR_3708""","""female""",58,"""IV""","""M1A""","""normal""",13.2,null,"""no""","""no""","""Yan et al.""",2,0,0,0,1,0,0,0,0,1,0,0,"""03660598B""","""Yan et al.""",2,6.570555,5.158157,4.697329,5.436335,5.282412,5.724922,4.673865,4.685542,5.002026,4.823392,5.520698,…,4.716858,4.614575,5.155613,4.738068,5.780937,6.69161,6.832015,6.412755,7.987326,4.711399,7.580741,4.09871,6.617454,6.15428,4.826822,3.883233,7.394174,4.58168,4.655143,5.489278,4.252246,4.676489,4.327306,6.693238,5.153024,7.247881,7.073525,4.00089,6.865694,6.033802,4.041276,4.613856,6.862496,5.049185,6.070983,5.128801,6.472128
"""YR_3705""","""male""",45,"""IV""","""M1A""","""normal""",17.4,null,"""no""","""no""","""Yan et al.""",2,0,0,0,1,0,0,0,0,1,0,0,"""03660602B""","""Yan et al.""",2,5.121619,5.6063,4.679799,4.94371,5.286963,5.540081,4.605398,4.685542,4.773099,4.822577,5.648834,…,5.567704,4.596732,5.128214,4.739978,5.847631,6.665335,6.35116,6.302636,6.596889,4.659869,7.471207,4.131287,6.622418,6.042509,4.929768,3.883233,5.551979,4.674432,4.639448,5.387564,4.281811,4.930001,4.372685,6.414757,4.831788,6.119016,6.675007,4.00089,5.629798,5.75064,4.011071,4.651746,6.769417,5.008392,5.894473,5.628744,6.725773
"""YR_1106""","""male""",48,"""IIIC""",null,"""normal""",1.4,null,"""no""","""no""","""Yan et al.""",0,1,0,0,1,0,0,0,0,1,0,0,"""04240076F""","""Yan et al.""",0,4.330612,5.12214,4.706186,4.544665,5.274799,5.671164,4.600569,4.695826,4.710835,4.824523,5.517643,…,4.717225,4.592083,5.105555,4.8011,5.686645,7.041298,6.355911,6.347053,7.068522,4.654083,7.454041,4.03427,6.614395,5.996841,4.839008,3.884491,5.007876,5.19275,4.509781,5.330479,4.249143,4.624107,4.32751,6.445733,4.836819,5.138535,6.479098,4.00089,4.951684,5.756891,4.040749,4.414295,6.52239,5.060871,5.74085,5.146984,6.30193
"""YR_1110""","""male""",33,"""IV""","""M1C""","""normal""",1.4,null,"""no""","""no""","""Yan et al.""",0,1,0,0,1,0,0,0,0,1,0,0,"""04240106F""","""Yan et al.""",0,4.330576,5.133844,4.745812,4.86601,5.278892,5.681413,4.888742,4.685542,4.86269,4.821645,5.376002,…,6.675318,4.600331,5.152749,4.811936,5.735904,7.155997,6.349381,6.461865,7.427956,4.675325,8.142162,4.262057,6.616135,6.289311,5.032443,3.883233,6.599731,4.782657,4.79579,5.453684,4.234814,4.724571,4.317788,6.498996,4.887051,6.686782,6.614028,4.00089,6.26734,5.995879,4.037211,4.800719,6.959851,5.064195,5.65068,5.206566,6.41538
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HL_Shi-35""","""male""",61,"""IV""","

In [87]:
clinical_gex.write_csv(f"../dataset/created/clinical_gex.csv")